# 🤖 Automated Machine Learning Pipeline

## Overview
This notebook implements a comprehensive automated machine learning (AutoML) pipeline for Ethiopian salary prediction. The pipeline automatically handles data preprocessing, feature engineering, model selection, hyperparameter tuning, and model evaluation.

### AutoML Components:
- 🔄 **Automated Data Preprocessing**: Missing values, encoding, scaling
- 🎨 **Automated Feature Engineering**: Feature creation and selection
- 🤖 **Automated Model Selection**: Algorithm comparison and selection
- ⚡ **Automated Hyperparameter Tuning**: Optimization algorithms
- 📊 **Automated Model Evaluation**: Comprehensive metrics and validation
- 🚀 **Automated Deployment**: Model packaging and serving

### Advanced Features:
- 🧠 **Meta-Learning**: Learning from previous experiments
- 🎯 **Multi-Objective Optimization**: Balancing accuracy and efficiency
- 📈 **Progressive Training**: Iterative model improvement
- 🔍 **Automated Interpretability**: Model explanation generation

---

In [ ]:
# Import comprehensive AutoML libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Core ML libraries
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_regression, RFE
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Model libraries
from sklearn.ensemble import (
    RandomForestRegressor, GradientBoostingRegressor, 
    AdaBoostRegressor, ExtraTreesRegressor
)
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet, 
    BayesianRidge, HuberRegressor
)
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor

# AutoML libraries
try:
    from auto_ml import Predictor
    AUTO_ML_AVAILABLE = True
    print("✅ auto_ml available for automated machine learning")
except ImportError:
    AUTO_ML_AVAILABLE = False
    print("⚠️ auto_ml not available. Install with: pip install auto_ml")

try:
    import autosklearn.regression
    AUTOSKLEARN_AVAILABLE = True
    print("✅ auto-sklearn available for automated ML")
except ImportError:
    AUTOSKLEARN_AVAILABLE = False
    print("⚠️ auto-sklearn not available. Install with: pip install auto-sklearn")

try:
    import tpot
    from tpot import TPOTRegressor
    TPOT_AVAILABLE = True
    print("✅ TPOT available for genetic programming AutoML")
except ImportError:
    TPOT_AVAILABLE = False
    print("⚠️ TPOT not available. Install with: pip install tpot")

try:
    import h2o
    from h2o.automl import H2OAutoML
    H2O_AVAILABLE = True
    print("✅ H2O AutoML available")
except ImportError:
    H2O_AVAILABLE = False
    print("⚠️ H2O not available. Install with: pip install h2o")

# Hyperparameter optimization
try:
    import optuna
    OPTUNA_AVAILABLE = True
    print("✅ Optuna available for hyperparameter optimization")
except ImportError:
    OPTUNA_AVAILABLE = False
    print("⚠️ Optuna not available. Install with: pip install optuna")

# Utilities
import time
import joblib
import json
from datetime import datetime
from IPython.display import display, HTML
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Set random seed
np.random.seed(42)

print("\n🤖 AutoML Setup Complete!")
print(f"🔄 auto_ml: {AUTO_ML_AVAILABLE}")
print(f"🧠 auto-sklearn: {AUTOSKLEARN_AVAILABLE}")
print(f"🧬 TPOT: {TPOT_AVAILABLE}")
print(f"💧 H2O AutoML: {H2O_AVAILABLE}")
print(f"🎯 Optuna: {OPTUNA_AVAILABLE}")
print("🚀 Ready for Automated Machine Learning!")

In [ ]:
# Custom AutoML Pipeline Implementation
print("🏗️ BUILDING CUSTOM AUTOML PIPELINE")
print("=" * 50)

class EthiopianSalaryAutoML:
    """
    Custom AutoML Pipeline for Ethiopian Salary Prediction
    
    This class implements a comprehensive automated machine learning pipeline
    specifically designed for salary prediction tasks.
    """
    
    def __init__(self, time_budget=300, cv_folds=5, random_state=42):
        self.time_budget = time_budget  # Time budget in seconds
        self.cv_folds = cv_folds
        self.random_state = random_state
        self.results = {}
        self.best_model = None
        self.best_score = float('inf')
        self.preprocessor = None
        self.feature_names = None
        
        # Define model candidates
        self.model_candidates = {
            'LinearRegression': LinearRegression(),
            'Ridge': Ridge(random_state=random_state),
            'Lasso': Lasso(random_state=random_state),
            'ElasticNet': ElasticNet(random_state=random_state),
            'RandomForest': RandomForestRegressor(random_state=random_state),
            'GradientBoosting': GradientBoostingRegressor(random_state=random_state),
            'ExtraTrees': ExtraTreesRegressor(random_state=random_state),
            'SVR': SVR(),
            'KNeighbors': KNeighborsRegressor(),
            'MLP': MLPRegressor(random_state=random_state, max_iter=500)
        }
        
        # Define hyperparameter grids
        self.param_grids = {
            'Ridge': {'alpha': [0.1, 1.0, 10.0, 100.0]},
            'Lasso': {'alpha': [0.1, 1.0, 10.0, 100.0]},
            'ElasticNet': {'alpha': [0.1, 1.0, 10.0], 'l1_ratio': [0.1, 0.5, 0.9]},
            'RandomForest': {
                'n_estimators': [50, 100, 200],
                'max_depth': [5, 10, None],
                'min_samples_split': [2, 5, 10]
            },
            'GradientBoosting': {
                'n_estimators': [50, 100, 200],
                'learning_rate': [0.01, 0.1, 0.2],
                'max_depth': [3, 5, 7]
            },
            'SVR': {
                'C': [0.1, 1, 10],
                'gamma': ['scale', 'auto', 0.01, 0.1],
                'kernel': ['rbf', 'linear']
            },
            'KNeighbors': {
                'n_neighbors': [3, 5, 7, 10],
                'weights': ['uniform', 'distance']
            },
            'MLP': {
                'hidden_layer_sizes': [(50,), (100,), (50, 50), (100, 50)],
                'alpha': [0.0001, 0.001, 0.01]
            }
        }
    
    def preprocess_data(self, X, y=None, fit=True):
        """Automated data preprocessing"""
        print("🔧 Automated Data Preprocessing")
        print("-" * 35)
        
        if fit:
            # Identify feature types
            numerical_features = X.select_dtypes(include=[np.number]).columns.tolist()
            categorical_features = X.select_dtypes(include=['object']).columns.tolist()
            
            print(f"📊 Numerical features: {len(numerical_features)}")
            print(f"🏷️  Categorical features: {len(categorical_features)}")
            
            # Create preprocessing pipeline
            transformers = []
            
            if numerical_features:
                transformers.append(('num', StandardScaler(), numerical_features))
            
            if categorical_features:
                transformers.append(('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), categorical_features))
            
            self.preprocessor = ColumnTransformer(transformers=transformers)
            X_processed = self.preprocessor.fit_transform(X)
            
            # Get feature names
            feature_names = []
            if numerical_features:
                feature_names.extend(numerical_features)
            if categorical_features:
                cat_feature_names = self.preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features)
                feature_names.extend(cat_feature_names)
            
            self.feature_names = feature_names
            print(f"✅ Preprocessing pipeline created: {X_processed.shape[1]} features")
            
        else:
            X_processed = self.preprocessor.transform(X)
        
        return X_processed
    
    def automated_feature_engineering(self, X, y):
        """Automated feature engineering and selection"""
        print("\n🎨 Automated Feature Engineering")
        print("-" * 35)
        
        # Feature selection using statistical tests
        selector = SelectKBest(score_func=f_regression, k='all')
        X_selected = selector.fit_transform(X, y)
        
        # Get feature scores
        feature_scores = selector.scores_
        feature_importance = pd.DataFrame({
            'feature': self.feature_names,
            'score': feature_scores
        }).sort_values('score', ascending=False)
        
        print(f"📊 Feature importance calculated")
        print(f"🎯 Top 5 features:")
        for i, (_, row) in enumerate(feature_importance.head().iterrows()):
            print(f"   {i+1}. {row['feature']}: {row['score']:.2f}")
        
        return X_selected, feature_importance
    
    def automated_model_selection(self, X, y):
        """Automated model selection and hyperparameter tuning"""
        print("\n🤖 Automated Model Selection")
        print("-" * 35)
        
        start_time = time.time()
        model_results = []
        
        for model_name, model in self.model_candidates.items():
            if time.time() - start_time > self.time_budget:
                print(f"⏰ Time budget exceeded. Stopping at {model_name}")
                break
            
            print(f"🔍 Testing {model_name}...")
            
            try:
                # Hyperparameter tuning if grid is defined
                if model_name in self.param_grids:
                    grid_search = GridSearchCV(
                        model, self.param_grids[model_name],
                        cv=self.cv_folds, scoring='neg_mean_squared_error',
                        n_jobs=-1, verbose=0
                    )
                    grid_search.fit(X, y)
                    best_model = grid_search.best_estimator_
                    best_score = -grid_search.best_score_
                    best_params = grid_search.best_params_
                else:
                    # Simple cross-validation
                    cv_scores = cross_val_score(
                        model, X, y, cv=self.cv_folds, 
                        scoring='neg_mean_squared_error'
                    )
                    best_model = model
                    best_score = -cv_scores.mean()
                    best_params = {}
                
                # Store results
                model_results.append({
                    'model_name': model_name,
                    'cv_rmse': np.sqrt(best_score),
                    'best_params': best_params,
                    'model': best_model
                })
                
                # Update best model
                if best_score < self.best_score:
                    self.best_score = best_score
                    self.best_model = best_model
                
                print(f"   ✅ CV RMSE: {np.sqrt(best_score):,.0f} ETB")
                
            except Exception as e:
                print(f"   ❌ Failed: {str(e)}")
                continue
        
        # Sort results by performance
        model_results.sort(key=lambda x: x['cv_rmse'])
        self.results['model_comparison'] = model_results
        
        print(f"\n🏆 Best model: {model_results[0]['model_name']}")
        print(f"🎯 Best CV RMSE: {model_results[0]['cv_rmse']:,.0f} ETB")
        
        return model_results
    
    def fit(self, X, y):
        """Fit the AutoML pipeline"""
        print("🚀 STARTING AUTOML PIPELINE")
        print("=" * 40)
        
        pipeline_start = time.time()
        
        # Step 1: Preprocess data
        X_processed = self.preprocess_data(X, y, fit=True)
        
        # Step 2: Feature engineering
        X_engineered, feature_importance = self.automated_feature_engineering(X_processed, y)
        self.results['feature_importance'] = feature_importance
        
        # Step 3: Model selection
        model_results = self.automated_model_selection(X_engineered, y)
        
        # Step 4: Final model training
        print("\n🎯 Training Final Model")
        print("-" * 25)
        self.best_model.fit(X_engineered, y)
        
        pipeline_time = time.time() - pipeline_start
        self.results['pipeline_time'] = pipeline_time
        
        print(f"\n✅ AutoML Pipeline Complete!")
        print(f"⏱️  Total time: {pipeline_time:.1f} seconds")
        print(f"🏆 Best model: {model_results[0]['model_name']}")
        
        return self
    
    def predict(self, X):
        """Make predictions using the best model"""
        X_processed = self.preprocess_data(X, fit=False)
        # Apply same feature engineering as training
        selector = SelectKBest(score_func=f_regression, k='all')
        # Note: In production, you'd save the fitted selector
        return self.best_model.predict(X_processed)
    
    def get_results_summary(self):
        """Get comprehensive results summary"""
        return self.results

print("✅ Custom AutoML Pipeline Implementation Complete!")
print("🤖 EthiopianSalaryAutoML class ready for use")